# Notebook 1 — Dataset Construction

Builds the instruction-tuning dataset from AWS docs + synthetic examples.

**Output:** `data/processed/train.jsonl` and `data/processed/test.jsonl`

In [ ]:
import sys
sys.path.append('../src')
from data_utils import save_jsonl
import json

## Step 1 — Seed examples (manual)

Start with 10-20 hand-crafted examples to establish the pattern.

In [ ]:
seed_examples = [
    {
        "instruction": "Generate an AWS IAM policy for the following requirement",
        "input": "The finance team needs read-only access to the S3 bucket 'prod-invoices'. No write or delete. MFA required.",
        "output": {
            "policy": {
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": ["s3:GetObject", "s3:ListBucket"],
                    "Resource": [
                        "arn:aws:s3:::prod-invoices",
                        "arn:aws:s3:::prod-invoices/*"
                    ],
                    "Condition": {
                        "Bool": {"aws:MultiFactorAuthPresent": "true"}
                    }
                }]
            },
            "nist_controls": ["AC-3", "AC-6", "IA-3"],
            "risk_notes": "Least-privilege applied. MFA enforced. No wildcard actions."
        }
    }
    # TODO: add more seed examples
]

print(f"{len(seed_examples)} seed examples loaded")

## Step 2 — Synthetic generation with GPT-4

Use seed examples as few-shot prompt to generate more examples.

In [ ]:
# TODO: call OpenAI API to generate synthetic examples
# Use seed_examples as few-shot context
# Review outputs manually before saving
synthetic_examples = []

## Step 3 — Combine, deduplicate, split

In [ ]:
from sklearn.model_selection import train_test_split

all_examples = seed_examples + synthetic_examples
train, test = train_test_split(all_examples, test_size=0.1, random_state=42)

save_jsonl(train, '../data/processed/train.jsonl')
save_jsonl(test, '../data/processed/test.jsonl')

print(f"Train: {len(train)} | Test: {len(test)}")